In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/metro_interstate_traffic_volume.csv")
df.shape

(48204, 9)

In [2]:
df['date_time']= pd.to_datetime(df['date_time'])
df.dtypes['date_time']

dtype('<M8[us]')

In [3]:
df=df.sort_values('date_time').reset_index(drop=True)
df.head()

,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,NaN,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,NaN,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,NaN,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,NaN,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,NaN,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


In [4]:
df['holiday'] = df['holiday'].fillna('None')
df['holiday'].isnull().sum()

np.int64(0)

In [5]:
print("Rows with temp = 0:", (df['temp'] == 0).sum())
df[df['temp'] == 0]

Rows with temp = 0: 10


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
11898,None,0.0,0.0,0.0,0,Clear,sky is clear,2014-01-31 03:00:00,361
11899,None,0.0,0.0,0.0,0,Clear,sky is clear,2014-01-31 04:00:00,734
11900,None,0.0,0.0,0.0,0,Clear,sky is clear,2014-01-31 05:00:00,2557
11901,None,0.0,0.0,0.0,0,Clear,sky is clear,2014-01-31 06:00:00,5150
11946,None,0.0,0.0,0.0,0,Clear,sky is clear,2014-02-02 03:00:00,291
11947,None,0.0,0.0,0.0,0,Clear,sky is clear,2014-02-02 04:00:00,284
11948,None,0.0,0.0,0.0,0,Clear,sky is clear,2014-02-02 05:00:00,434
11949,None,0.0,0.0,0.0,0,Clear,sky is clear,2014-02-02 06:00:00,739
11950,None,0.0,0.0,0.0,0,Clear,sky is clear,2014-02-02 07:00:00,962
11951,None,0.0,0.0,0.0,0,Clear,sky is clear,2014-02-02 08:00:00,1670


In [6]:
df['temp'] = df['temp'].replace(0, np.nan)
df['temp'] = df['temp'].interpolate()
print("Remaining zeros:", (df['temp'] == 0).sum())

Remaining zeros: 0


In [7]:
print("Rows with rain_1h >300:" ,(df['rain_1h']>300).sum())
df[df['rain_1h']>300]

Rows with rain_1h >300: 1


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
24872,None,302.11,9831.3,0.0,75,Rain,very heavy rain,2016-07-11 17:00:00,5535


In [8]:
df['rain_1h'] = df['rain_1h'].apply(lambda x: 0 if x > 300 else x)
df['rain_1h'].max()

np.float64(55.63)

In [9]:
print("Before:", df.shape)
df = df.drop_duplicates()
print("After:", df.shape)

Before: (48204, 9)
After: (48187, 9)


In [10]:
dup_rows = df[df.duplicated(subset='date_time', keep=False)]
dup_rows.groupby('date_time')['traffic_volume'].nunique().value_counts()

traffic_volume
1    5430
Name: count, dtype: int64

In [11]:
print("Before:", df.shape)
df = df.drop_duplicates(subset='date_time', keep='first')
print("After:", df.shape)

Before: (48187, 9)
After: (40575, 9)


In [12]:
df = df.drop(columns=['weather_description'])
df.columns.tolist()

['holiday',
 'temp',
 'rain_1h',
 'snow_1h',
 'clouds_all',
 'weather_main',
 'date_time',
 'traffic_volume']

In [13]:
print(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())
print("date_time all unique:", df['date_time'].is_unique)
df.describe()

holiday           0
temp              0
rain_1h           0
snow_1h           0
clouds_all        0
weather_main      0
date_time         0
traffic_volume    0
dtype: int64
Duplicate rows: 0
date_time all unique: True


,temp,rain_1h,snow_1h,clouds_all,date_time,traffic_volume
count,40575.000000,40575.000000,40575.000000,40575.000000,40575,40575.000000
mean,281.379371,0.076306,0.000117,44.195835,2015-12-23 22:16:28.835489,3290.650474
min,243.390000,0.000000,0.000000,0.000000,2012-10-02 09:00:00,0.000000
25%,271.840000,0.000000,0.000000,1.000000,2014-02-02 19:30:00,1248.500000
50%,282.860000,0.000000,0.000000,40.000000,2016-06-02 14:00:00,3427.000000
75%,292.280000,0.000000,0.000000,90.000000,2017-08-02 23:30:00,4952.000000
max,310.070000,55.630000,0.510000,100.000000,2018-09-30 23:00:00,7280.000000
std,13.098215,0.769547,0.005676,38.684567,NaN,1984.772909


In [14]:
df.to_csv('../data/processed/metro_interstate_traffic_volume_clean.csv', index=False)
print("Saved:", df.shape)

Saved: (40575, 8)
